## display(dbutils.fs.ls('caminho')


Mostra todos os arquivos que estão dentro do Volume chamado "dados" que foi criado dentro do catálogo workspace, schema/database default.

In [0]:
display(dbutils.fs.ls('/'))

path,name,size,modificationTime
dbfs:/Volumes/,Volumes/,0,0
dbfs:/Workspace/,Workspace/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0


In [0]:
display(dbutils.fs.ls('/Volumes/workspace/landing/dados/'))

path,name,size,modificationTime
dbfs:/Volumes/workspace/landing/dados/games.csv,games.csv,2412383,1792523447000
dbfs:/Volumes/workspace/landing/dados/how_long_to_beat.csv,how_long_to_beat.csv,296219,1792523449000
dbfs:/Volumes/workspace/landing/dados/regional_sales.csv,regional_sales.csv,507445,1792523450000
dbfs:/Volumes/workspace/landing/dados/steam_metrics.csv,steam_metrics.csv,106183,1792523452000
dbfs:/Volumes/workspace/landing/dados/walkthrough_links.csv,walkthrough_links.csv,47470,1792523454000


## df = spark.read.option("infeschema", "true").option("header", "true").csv("path + filename")

Gera um dataframe para cada arquivo que está no Volume "dados".


In [0]:
landing_path = '/Volumes/workspace/landing/dados'

df_games    = spark.read.option("infeschema", "true").option("header", "true").csv(f"{landing_path}/games.csv")
df_steam    = spark.read.option("infeschema", "true").option("header", "true").csv(f"{landing_path}/steam_metrics.csv")
df_sales    = spark.read.option("infeschema", "true").option("header", "true").csv(f"{landing_path}/regional_sales.csv")
df_hltob    = spark.read.option("infeschema", "true").option("header", "true").csv(f"{landing_path}/how_long_to_beat.csv")
df_links    = spark.read.option("infeschema", "true").option("header", "true").csv(f"{landing_path}/walkthrough_links.csv")


## df = df_games.withColumn("nome_coluna", "valor")

Adiciona uma nova coluna (metadado) de data e hora de processamento e nome do arquivo de origem.

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_games     = df_games.withColumn("datetime_bronze", current_timestamp()).withColumn("filename", lit("games.csv"))
df_steam     = df_steam.withColumn("datetime_bronze", current_timestamp()).withColumn("filename", lit("steam_metrics.csv"))
df_sales     = df_sales.withColumn("datetime_bronze", current_timestamp()).withColumn("filename", lit("regional_sales.csv"))
df_hltob     = df_hltob.withColumn("datetime_bronze", current_timestamp()).withColumn("filename", lit("how_long_to_beat.csv"))
df_links     = df_links.withColumn("datetime_bronze", current_timestamp()).withColumn("filename", lit("walkthrough_links.csv"))

df_games.show(5)

+---+---------------+-----------------+------------+---------+----------+--------------------+--------------------+--------------------+--------------------+---------+
| id|           name|         platform|release_date|metascore|user_score|           developer|           publisher|               genre|     datetime_bronze| filename|
+---+---------------+-----------------+------------+---------+----------+--------------------+--------------------+--------------------+--------------------+---------+
|  1|          #BLUD|               PC|  2024-06-18|       73|       5.6|     Exit 73 Studios|    ['Humble Games']|    Action Adventure|2026-10-20 19:26:...|games.csv|
|  2|         #DRIVE|iOS (iPhone/iPad)|  2021-02-16|       69|       6.8|    Dariusz Pietrala|['PM Studios Inc....|         Auto Racing|2026-10-20 19:26:...|games.csv|
|  3|         #IDARB|         Xbox One|  2015-01-30|       77|       6.9|Other Ocean Inter...|['Other Ocean Int...|               Party|2026-10-20 19:26:...|gam

## df_games.write.format('delta').saveAsTable("nome_tabela")

Salva os dataframes em arquivos delta lake (formato de arquivo) no schema/database "bronze". As tabelas geradas são do tipo MANAGED (gerenciadas).

In [0]:
df_games.write.format('delta').mode("overwrite").saveAsTable("bronze.metacritic_games")
df_steam.write.format('delta').mode("overwrite").saveAsTable("bronze.steam_metrics")
df_sales.write.format('delta').mode("overwrite").saveAsTable("bronze.regional_sales")
df_hltob.write.format('delta').mode("overwrite").saveAsTable("bronze.how_long_to_beat")
df_links.write.format('delta').mode("overwrite").saveAsTable("bronze.walkthrough_links")

## (SQL) SHOW TABLES IN bronze

Verifica os dados gravados no formato delta lake tipo MANAGED na camada bronze.

In [0]:
%sql
SHOW TABLES IN bronze

database,tableName,isTemporary
bronze,how_long_to_beat,false
bronze,metacritic_games,false
bronze,regional_sales,false
bronze,steam_metrics,false
bronze,walkthrough_links,false


## (SQL) DESCRIBE DETAIL nome_tabela;


Vendo os detalhes de um tabela delta lake.

In [0]:
%sql
DESCRIBE DETAIL bronze.metacritic_games;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,6a1e128e-b6f9-43fa-bf8f-e666dddf230f,workspace.bronze.metacritic_games,null,,2026-10-20T19:26:57.235Z,2026-10-20T19:27:02.000Z,List(),List(),1,439694,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> b7760a4a-8c88-48db-acfc-ae1a6ff1a508, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-62c16e15-b6f0-4f81-881b-39be3a4e810c, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-d55e8e59-3caf-41ff-b7be-69f079e08000)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


## (SQL) DESCRIBE EXTENDED nome_tabela;
ou 
##(SQL) DESCRIBE TABLE EXTENDED nome_tabela;

Mostra se a tabela é MANAGED Ou EXTERNAL.


In [0]:
%sql
DESCRIBE EXTENDED bronze.metacritic_games;
--DESCRIBE TABLE EXTENDED apolice_bronze;

col_name,data_type,comment
id,string,null
name,string,null
platform,string,null
release_date,string,null
metascore,string,null
user_score,string,null
developer,string,null
publisher,string,null
genre,string,null
datetime_bronze,timestamp,null
